# ORBIT First Tool Call Session Demo

This notebook now covers both the provider-native capability probe and the first ORBIT tool-call closure path on SSH vLLM.


## Probe design

This notebook checks the following layers:
1. Canonical prompt state
2. Provider-native payload shape
3. Raw HTTP response body
4. Parsed provider message / tool_calls / finish_reason
5. ORBIT normalized interpretation
6. ORBIT closure path (tool execution + transcript reinjection + final answer)


In [1]:
from pathlib import Path
import requests

from orbit.models import ConversationMessage, MessageRole
from orbit.runtime import OrbitCoordinator
from orbit.runtime.providers.openai_codex import OpenAICodexConfig, OpenAICodexExecutionBackend
from orbit.store import create_default_store
from orbit.notebook import session_messages_dataframe
from orbit.notebook.providers.tool_call_debug import (
    build_tool_call_probe_backend,
    probe_tool_call_request,
    probe_codex_tool_call_raw_events,
    resolve_probe_base_url,
    run_tool_call_approval_demo,
    run_tool_call_closure,
    tool_call_approval_summary_frame,
    tool_call_closure_summary_frame,
    tool_call_probe_summary_frame,
    codex_raw_events_frame,
    probe_rejection_continuation_context,
    continuation_context_summary_frame,
    continuation_context_payload,
)

REPO_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().resolve().name == 'providers' else Path.cwd().resolve()
API_URL = 'http://127.0.0.1:8000/v1/chat/completions'
MODEL = 'Salesforce/Llama-xLAM-2-8b-fc-r'
TEST_PROMPTS = [
    'Use the read_file tool to read notes/scaffold/demo.txt. Do not answer from memory. After using the tool, tell me the first sentence only.',
    'You must use the read_file tool before answering. If you do not use the tool, your answer is incorrect. Read notes/scaffold/demo.txt and return only the first sentence.',
    'Use the read_file tool with path="notes/scaffold/demo.txt". Do not answer without calling the tool first.',
]
PROMPT = TEST_PROMPTS[0]
APPROVAL_PROMPT = 'Use the write_file tool to write notes/approval-demo.txt with the exact content: Approval demo written by ORBIT. After using the tool, confirm what you wrote.'
backend = build_tool_call_probe_backend(
    workspace_root=REPO_ROOT,
    remote_base_url='http://127.0.0.1:8000/v1',
    model=MODEL,
    api_key='EMPTY',
    auto_tunnel=True,
    ssh_host='ws3',
)
messages = [ConversationMessage(session_id='session:tool-probe', role=MessageRole.USER, content=PROMPT, turn_index=1)]
payload = backend.build_request_payload_from_messages(messages)
READY_BASE_URL = resolve_probe_base_url(backend=backend)
READY_API_URL = READY_BASE_URL.rstrip('/') + '/chat/completions'


## 1. Canonical prompt state


In [2]:
PROMPT

'Use the read_file tool to read notes/scaffold/demo.txt. Do not answer from memory. After using the tool, tell me the first sentence only.'

## 2. Provider-assembled payload


In [3]:
payload

{'model': 'Salesforce/Llama-xLAM-2-8b-fc-r',
 'messages': [{'role': 'user',
   'content': 'Use the read_file tool to read notes/scaffold/demo.txt. Do not answer from memory. After using the tool, tell me the first sentence only.'}],
 'stream': False,
 'tools': [{'type': 'function',
   'function': {'name': 'read_file',
    'description': 'Read a file from the ORBIT workspace.',
    'parameters': {'type': 'object',
     'properties': {'path': {'type': 'string',
       'description': 'Workspace-relative file path.'}},
     'required': ['path']}}},
  {'type': 'function',
   'function': {'name': 'write_file',
    'description': 'Write text content to a file in the ORBIT workspace.',
    'parameters': {'type': 'object',
     'properties': {'path': {'type': 'string',
       'description': 'Workspace-relative file path.'},
      'content': {'type': 'string', 'description': 'File content to write.'}},
     'required': ['path', 'content']}}}],
 'tool_choice': 'auto'}

## 3. Raw provider-native HTTP response


In [4]:
response = requests.post(
    READY_API_URL,
    headers={
        'Authorization': 'Bearer EMPTY',
        'Content-Type': 'application/json',
    },
    json=payload,
    timeout=60,
)
response.status_code


200

In [5]:
raw_text = response.text
raw_text[:4000]


'{"id":"chatcmpl-a68052736beb01be","object":"chat.completion","created":1775120211,"model":"Salesforce/Llama-xLAM-2-8b-fc-r","choices":[{"index":0,"message":{"role":"assistant","content":null,"refusal":null,"annotations":null,"audio":null,"function_call":null,"tool_calls":[{"id":"call_0_87c8d0a67a92e2c0","type":"function","function":{"name":"read_file","arguments":"{\\"path\\": \\"notes/scaffold/demo.txt\\"}"}}],"reasoning":null},"logprobs":null,"finish_reason":"tool_calls","stop_reason":null,"token_ids":null}],"service_tier":null,"system_fingerprint":null,"usage":{"prompt_tokens":420,"total_tokens":443,"completion_tokens":23,"prompt_tokens_details":null},"prompt_logprobs":null,"prompt_token_ids":null,"kv_transfer_params":null}'

## 4. Parsed provider message / tool_calls / finish_reason


In [6]:
data = response.json()
data


{'id': 'chatcmpl-a68052736beb01be',
 'object': 'chat.completion',
 'created': 1775120211,
 'model': 'Salesforce/Llama-xLAM-2-8b-fc-r',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': None,
    'refusal': None,
    'annotations': None,
    'audio': None,
    'function_call': None,
    'tool_calls': [{'id': 'call_0_87c8d0a67a92e2c0',
      'type': 'function',
      'function': {'name': 'read_file',
       'arguments': '{"path": "notes/scaffold/demo.txt"}'}}],
    'reasoning': None},
   'logprobs': None,
   'finish_reason': 'tool_calls',
   'stop_reason': None,
   'token_ids': None}],
 'service_tier': None,
 'system_fingerprint': None,
 'usage': {'prompt_tokens': 420,
  'total_tokens': 443,
  'completion_tokens': 23,
  'prompt_tokens_details': None},
 'prompt_logprobs': None,
 'prompt_token_ids': None,
 'kv_transfer_params': None}

In [7]:
choice = data['choices'][0]
{
    'finish_reason': choice.get('finish_reason'),
    'message': choice.get('message'),
}


{'finish_reason': 'tool_calls',
 'message': {'role': 'assistant',
  'content': None,
  'refusal': None,
  'annotations': None,
  'audio': None,
  'function_call': None,
  'tool_calls': [{'id': 'call_0_87c8d0a67a92e2c0',
    'type': 'function',
    'function': {'name': 'read_file',
     'arguments': '{"path": "notes/scaffold/demo.txt"}'}}],
  'reasoning': None}}

## 5. ORBIT normalized interpretation


In [8]:
probe_result = probe_tool_call_request(backend=backend, prompt=PROMPT)
probe_result['ready_base_url'], tool_call_probe_summary_frame(probe_result)


('http://127.0.0.1:8000/v1',
              ready_base_url             plan_label source_backend  \
 0  http://127.0.0.1:8000/v1  ssh-vllm-tool-request       ssh-vllm   
 
    has_tool_request  tool_name  requires_approval side_effect_class  \
 0              True  read_file              False              safe   
 
   failure_reason  
 0           None  )

In [9]:
probe_result['plan'].model_dump()


{'source_backend': 'ssh-vllm',
 'plan_label': 'ssh-vllm-tool-request',
 'final_text': None,
 'tool_request': {'tool_name': 'read_file',
  'input_payload': {'path': 'notes/scaffold/demo.txt'},
  'requires_approval': False,
  'side_effect_class': 'safe'},
 'should_finish_after_tool': False,
 'failure_reason': None}

## 6. ORBIT closure path (tool execution + transcript reinjection + final answer)


In [10]:
closure_result = run_tool_call_closure(workspace_root=REPO_ROOT, backend=backend, prompt=PROMPT)
tool_call_closure_summary_frame(closure_result)


,session_id,plan_label,source_backend,final_text,message_count,last_message_role,last_message_kind
0,session_68b8b8dac5fd,ssh-vllm-final-text,ssh-vllm,The first sentence of the file notes/scaffold/...,3,assistant,None


In [11]:
session_messages_dataframe(closure_result['coordinator'].store, closure_result['session'].session_id)


,message_id,session_id,turn_index,role,content,created_at,provider_message_id,metadata
0,msg_24dbda77b029,session_68b8b8dac5fd,1,user,Use the read_file tool to read notes/scaffold/...,2026-04-02 08:57:13.765780+00:00,None,{}
1,msg_f53be071d122,session_68b8b8dac5fd,2,tool,Tool result from read_file:\nORBIT demo output\n,2026-04-02 08:57:14.956209+00:00,None,"{'message_kind': 'tool_result', 'tool_name': '..."
2,msg_782a1efe165e,session_68b8b8dac5fd,3,assistant,The first sentence of the file notes/scaffold/...,2026-04-02 08:57:15.978216+00:00,None,"{'source_backend': 'ssh-vllm', 'plan_label': '..."


In [13]:
closure_result['final_plan'].model_dump()


{'source_backend': 'ssh-vllm',
 'plan_label': 'ssh-vllm-tool-request',
 'final_text': None,
 'tool_request': {'tool_name': 'read_file',
  'input_payload': {'path': 'notes/scaffold/demo.txt'},
  'requires_approval': False,
  'side_effect_class': 'safe'},
 'should_finish_after_tool': False,
 'failure_reason': None}

## 7. Approval-gated closure path (write tool with explicit session approval)

This section exercises the new session-local approval path for a side-effecting tool.

Reference comparison note:
- ORBIT is currently keeping approval transcript-visible and session-inspectable.
- Claw Code / similar coding-agent runtimes lean more heavily on permission policy and tool gating modes.
- ORBIT may later combine both, but this notebook keeps approval as an explicit governed artifact in the inspected flow.


In [2]:
approval_result = run_tool_call_approval_demo(
    workspace_root=REPO_ROOT,
    backend=backend,
    prompt=APPROVAL_PROMPT,
    decision='approve',
    note='approved in first approval demo',
)
tool_call_approval_summary_frame(approval_result)


,session_id,waiting_plan_label,waiting_tool_name,approval_request_id,resolved_plan_label,resolved_final_text,message_count,last_message_role,last_message_kind
0,session_7dd2dd3c616d,ssh-vllm-tool-request-waiting-for-approval,write_file,approval_fca1263077f5,ssh-vllm-final-text,I have successfully written the content to the...,5,assistant,None


In [3]:
approval_result['open_approvals']


[{'session_id': 'session_7dd2dd3c616d',
  'conversation_id': 'conversation:ssh-vllm:1775120749',
  'approval_request_id': 'approval_fca1263077f5',
  'tool_request': {'tool_name': 'write_file',
   'input_payload': {'path': 'notes/approval-demo.txt',
    'content': 'Approval demo written by ORBIT.'},
   'requires_approval': True,
   'side_effect_class': 'write'},
  'source_backend': 'ssh-vllm',
  'plan_label': 'ssh-vllm-tool-request',
  'opened_at': '2026-04-02T09:05:51.058800+00:00'}]

In [4]:
approval_result['messages_df']


,message_id,session_id,turn_index,role,content,created_at,provider_message_id,metadata
0,msg_620a29c6b44b,session_7dd2dd3c616d,1,user,Use the write_file tool to write notes/approva...,2026-04-02 09:05:49.313043+00:00,None,{}
1,msg_bd8ddcb84bcc,session_7dd2dd3c616d,2,assistant,Approval required before executing write_file ...,2026-04-02 09:05:51.061686+00:00,None,"{'message_kind': 'approval_request', 'approval..."
2,msg_e20f9f746481,session_7dd2dd3c616d,3,assistant,Approval granted for write_file. Continuing wi...,2026-04-02 09:05:51.119628+00:00,None,"{'message_kind': 'approval_decision', 'approva..."
3,msg_c8cfa6af2d64,session_7dd2dd3c616d,4,tool,Tool result from write_file:\nwrote /Volumes/2...,2026-04-02 09:05:51.121355+00:00,None,"{'message_kind': 'tool_result', 'tool_name': '..."
4,msg_9f2f2a2d2b4e,session_7dd2dd3c616d,5,assistant,I have successfully written the content to the...,2026-04-02 09:05:52.486325+00:00,None,"{'source_backend': 'ssh-vllm', 'plan_label': '..."


## 8. Approval-rejected closure path (write tool blocked by explicit rejection)

This section shows the rejection branch of the same governed approval path.
The important validation signal is that the tool is not executed and the target file is not created.


In [5]:
reject_result = run_tool_call_approval_demo(
    workspace_root=REPO_ROOT,
    backend=backend,
    prompt='Use the write_file tool to write notes/approval-demo-reject.txt with the exact content: Approval reject demo written by ORBIT. After using the tool, confirm what you wrote.',
    decision='reject',
    note='rejected in notebook showcase',
)
tool_call_approval_summary_frame(reject_result)


,session_id,waiting_plan_label,waiting_tool_name,approval_request_id,resolved_plan_label,resolved_final_text,message_count,last_message_role,last_message_kind
0,session_f8972fae1e56,ssh-vllm-tool-request-waiting-for-approval,write_file,approval_02a8c5a6dc03,ssh-vllm-tool-request-rejected,Approval rejected for write_file. The requeste...,3,assistant,approval_decision


In [6]:
reject_result['messages_df']


,message_id,session_id,turn_index,role,content,created_at,provider_message_id,metadata
0,msg_43e683a536b9,session_f8972fae1e56,1,user,Use the write_file tool to write notes/approva...,2026-04-02 09:06:01.674042+00:00,None,{}
1,msg_f110e31d2b83,session_f8972fae1e56,2,assistant,Approval required before executing write_file ...,2026-04-02 09:06:03.550477+00:00,None,"{'message_kind': 'approval_request', 'approval..."
2,msg_26bd29333005,session_f8972fae1e56,3,assistant,Approval rejected for write_file. The requeste...,2026-04-02 09:06:03.556252+00:00,None,"{'message_kind': 'approval_decision', 'approva..."


In [7]:
(REPO_ROOT / 'notes' / 'approval-demo-reject.txt').exists()


False

## 9. OpenAI Codex raw SSE tool-call probe and first closure

This section shows the hosted Codex route at three levels: request payload, raw SSE events, and ORBIT-normalized interpretation.


In [4]:
codex_probe = probe_codex_tool_call_raw_events(
    workspace_root=REPO_ROOT,
    prompt='Use the read_file tool to read notes/scaffold/demo.txt. Do not answer from memory. After using the tool, tell me the first sentence only.',
)
codex_probe['normalized_plan'].model_dump()


{'source_backend': 'openai-codex',
 'plan_label': 'openai-codex-tool-request',
 'final_text': None,
 'tool_request': {'tool_name': 'read_file',
  'input_payload': {'path': 'notes/scaffold/demo.txt'},
  'requires_approval': False,
  'side_effect_class': 'safe'},
 'should_finish_after_tool': False,
 'failure_reason': None}

In [5]:
codex_raw_events_frame(codex_probe)


,index,type,raw_line,payload_json
0,0,response.created,"data: {""type"":""response.created"",""response"":{""...","{""type"": ""response.created"", ""response"": {""id""..."
1,1,response.in_progress,"data: {""type"":""response.in_progress"",""response...","{""type"": ""response.in_progress"", ""response"": {..."
2,2,response.output_item.added,"data: {""type"":""response.output_item.added"",""it...","{""type"": ""response.output_item.added"", ""item"":..."
3,3,response.function_call_arguments.delta,"data: {""type"":""response.function_call_argument...","{""type"": ""response.function_call_arguments.del..."
4,4,response.function_call_arguments.delta,"data: {""type"":""response.function_call_argument...","{""type"": ""response.function_call_arguments.del..."
5,5,response.function_call_arguments.delta,"data: {""type"":""response.function_call_argument...","{""type"": ""response.function_call_arguments.del..."
6,6,response.function_call_arguments.delta,"data: {""type"":""response.function_call_argument...","{""type"": ""response.function_call_arguments.del..."
7,7,response.function_call_arguments.delta,"data: {""type"":""response.function_call_argument...","{""type"": ""response.function_call_arguments.del..."
8,8,response.function_call_arguments.delta,"data: {""type"":""response.function_call_argument...","{""type"": ""response.function_call_arguments.del..."
9,9,response.function_call_arguments.delta,"data: {""type"":""response.function_call_argument...","{""type"": ""response.function_call_arguments.del..."


In [6]:
codex_backend = OpenAICodexExecutionBackend(
    config=OpenAICodexConfig(model='gpt-5.4'),
    repo_root=REPO_ROOT,
)
codex_coordinator = OrbitCoordinator(
    store=create_default_store(),
    workspace_root=REPO_ROOT,
    backend=codex_backend,
)
codex_session = codex_coordinator.create_session(backend_name='openai-codex', model='gpt-5.4')
codex_closure_plan = codex_coordinator.run_session_turn(
    session_id=codex_session.session_id,
    user_input='Use the read_file tool to read notes/scaffold/demo.txt. Do not answer from memory. After using the tool, tell me the first sentence only.',
)
codex_closure_plan.model_dump()


{'source_backend': 'openai-codex',
 'plan_label': 'openai-codex-final-text',
 'final_text': 'ORBIT demo output',
 'tool_request': None,
 'should_finish_after_tool': True,
 'failure_reason': None}

In [7]:
session_messages_dataframe(codex_coordinator.store, codex_session.session_id)


,message_id,session_id,turn_index,role,content,created_at,provider_message_id,metadata
0,msg_2a7cdfda217b,session_ea7c49625787,1,user,Use the read_file tool to read notes/scaffold/...,2026-04-02 10:41:09.204494+00:00,None,{}
1,msg_c872d066e14e,session_ea7c49625787,2,tool,Tool result from read_file:\nORBIT demo output\n,2026-04-02 10:41:10.393986+00:00,None,"{'message_kind': 'tool_result', 'tool_name': '..."
2,msg_ffa5e2c9a452,session_ea7c49625787,3,assistant,ORBIT demo output,2026-04-02 10:41:11.588039+00:00,None,"{'source_backend': 'openai-codex', 'plan_label..."


## 10. Continuation-context inspection for rejection handling

This section exposes the current continuation-context package separately from canonical transcript and provider payload layers.


In [2]:
rejection_messages = [
    ConversationMessage(
        session_id='session:continuation-context-demo',
        role=MessageRole.USER,
        content='Use the write_file tool to write notes/context-demo.txt with the exact content: Context demo written by ORBIT.',
        turn_index=1,
    ),
    ConversationMessage(
        session_id='session:continuation-context-demo',
        role=MessageRole.ASSISTANT,
        content='Approval required before executing write_file (side_effect_class=write).',
        turn_index=2,
        metadata={
            'message_kind': 'approval_request',
            'approval_request_id': 'approval_demo',
            'tool_name': 'write_file',
            'side_effect_class': 'write',
        },
    ),
    ConversationMessage(
        session_id='session:continuation-context-demo',
        role=MessageRole.ASSISTANT,
        content='Approval rejected for write_file. The requested tool was not executed. Note: rejected in continuation context demo',
        turn_index=3,
        metadata={
            'message_kind': 'approval_decision',
            'approval_request_id': 'approval_demo',
            'decision': 'rejected',
            'note': 'rejected in continuation context demo',
            'tool_name': 'write_file',
            'tool_executed': False,
        },
    ),
]
continuation_probe = probe_rejection_continuation_context(rejection_messages)
continuation_context_summary_frame(continuation_probe)


,context_kind,has_bridge_message,has_system_prompt,allowed_next_actions_count,tool_name
0,approval_rejection,True,True,3,write_file


In [3]:
continuation_context_payload(continuation_probe)


{'context_kind': 'approval_rejection',
 'bridge_message': {'message_id': 'msg_92ed295e8c9b',
  'session_id': 'session:continuation-context-demo',
  'role': 'assistant',
  'content': 'Continuation bridge: the previously requested tool `write_file` was rejected by governance and was not executed. Note: rejected in continuation context demo You must continue truthfully from that state. Do not claim that the tool ran or that its side effects occurred. Acknowledge the rejection and continue with a safe alternative, explanation, or next non-side-effecting step.',
  'turn_index': 3,
  'created_at': '2026-04-02T12:34:19.932674Z',
  'provider_message_id': None,
  'metadata': {'message_kind': 'continuation_bridge',
   'bridge_kind': 'approval_rejection',
   'tool_name': 'write_file',
   'tool_executed': False,
   'source_message_kind': 'approval_decision'}},
 'system_prompt': 'Governance constraint for this continuation: the previously requested tool write_file was rejected and must not be calle